This code verifies and understands the existing catalog parser code.


STEP 1: Initialization and loading network from XML file

In [1]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
container = feeder
network = FeederModel(container=feeder, connection=file)

STEP 2: Run Catalog Parser and get Object Output, this will serve as input to the New_New_Power_Transformer function

In [4]:
import logging
import json 
from cimgraph.models import GraphModel
from __future__ import annotations


from cimgraph.databases import get_cim_profile
_log = logging.getLogger(__name__)

def catalog_parser(catalog_file, network):
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile 
    obj = item_parser(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser(data:dict, network: GraphModel, cim):
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data:
        if type(data[attribute]) == str:
            if attribute in class_type.__dataclass_fields__:
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list:
            if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)
    return obj

Catalog_JSON_file_path = '../test_models/hv69_12.json' ### Power Transformer json file path
Obj_CP_output = catalog_parser(Catalog_JSON_file_path, network) ### Catalog file reads the input and returns an object 

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


Step 3: Save the updated network to xml file. 

In [7]:
from cimgraph import utils
utils.write_xml(network=network, filename='IEEE_13_with_Xfmr_CPOB.xml')

import os
print(os.getcwd())

/root/CIM-Builder/tests/test_scripts


STEP 3: Write sample code capturing the new_power_transformer function contents, and use the CP output object as input. This can later be modified as the new function

STEP 3a. Trial by simply modifying the existing new_power_transformer function

In [ ]:
# def terminal_to_node(network:GraphModel, terminal:cim.Terminal, node:str|cim.ConnectivityNode):
#     if node.__class__ == str:
#         for node_obj in network.graph[cim.ConnectivityNode].values():
#             if node_obj.name == node or node_obj.aliasName == node:
#                 terminal.ConnectivityNode = node_obj
#                 node_obj.Terminals.append(terminal)
#     else:
#         terminal.ConnectivityNode = node
#         node.Terminals.append(terminal)

In [ ]:
# name = 'Power_Transformer_XYZ'
# node_P_xfmr_1 = '646'
# node_P_xfmr_2 = '645'

# cim_profile, cim_module = get_cim_profile()
# cim:cim = cim_module

# #synchr_generator = cim.SynchronousMachine(name = name)

In [ ]:
# Catalog_JSON_file_path = '../test_models/hv69_12.json' ### Power Transformer json file path
# catalog_file = '../test_models/hv69_12.json' ### Power Transformer json file path
# if catalog_file is not None:
#     print('Catalog file has content')
#     xfmr_1 = catalog_parser(catalog_file, network) ## used Catalog Parser and created the output Object
#     xfmr_1.name = name
#     xfmr_1.EquipmentContainer = container

#     for end in xfmr_1.PowerTransformerEnd:
#         end.PowerTransformer = xfmr_1
#         number = int(end.endNumber) ## These numbers will be 1 and 2, which denotes the 1st and 2nd windings in the 2-winding transformer respectively
#         print(number)
#         terminal = cim.Terminal(name=f"{xfmr_1.name}_t{number}", sequenceNumber=number)
#         print(terminal)
#         end.Terminal = terminal
#         xfmr_1.Terminals.append(terminal)
#         print(xfmr_1.Terminals)
#         network.add_to_graph(terminal)

        
#         if number == 1:
#             terminal_to_node(network, terminal, node_P_xfmr_1)
#         if number == 2:
#             terminal_to_node(network, terminal, node_P_xfmr_2)

# print(xfmr_1)

# #print(network)

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


Catalog file has content
1
{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal", "name": "Power_Transformer_XYZ_t1", "sequenceNumber": "1"}
[{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal"}]
2
{"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal", "name": "Power_Transformer_XYZ_t2", "sequenceNumber": "2"}
[{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal"}, {"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal"}]
{"@id": "f5a479e5-6fae-4445-a2fb-4f4e7e9a7f43", "@type": "PowerTransformer", "name": "Power_Transformer_XYZ", "EquipmentContainer": {"@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62", "@type": "Feeder"}, "Terminals": [{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal"}, {"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal"}], "vectorGroup": "Yy", "PowerTransformerEnd": [{"@id": "f5b6b52e-2c52-4cc6-af5e-db766f912576", "@type": "PowerTransformerEnd"}, {"@id": "b83cd46e-36a0

In [ ]:
# from cimgraph import utils
# utils.write_xml(network=network, filename='IEEE_13_with_Xfmr_CPOB.xml')

In [ ]:
# print(xfmr_1)
# print(xfmr_1.__dict__)
# print(vars(xfmr_1))

# print(xfmr_1.PowerTransformerEnd[1])

{"@id": "3eac6265-b484-481e-94f8-0898133fd392", "@type": "PowerTransformer", "name": "Power_Transformer_XYZ", "EquipmentContainer": {"@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62", "@type": "Feeder"}, "Terminals": [{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal"}, {"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal"}], "vectorGroup": "Yy", "PowerTransformerEnd": [{"@id": "2ac990a4-262a-446e-b97b-d5ba561936ef", "@type": "PowerTransformerEnd"}, {"@id": "aaefa65e-dfc6-4a50-8896-de90b094849e", "@type": "PowerTransformerEnd"}]}
{'identifier': UUID('3eac6265-b484-481e-94f8-0898133fd392'), 'mRID': '3eac6265-b484-481e-94f8-0898133fd392', 'aliasName': None, 'description': None, 'name': 'Power_Transformer_XYZ', 'Names': [], 'AssetDatasheet': None, 'Assets': [], 'Controls': [], 'Location': None, 'Measurements': [], 'PSRType': None, 'aggregate': None, 'inService': None, 'networkAnalysisEnabled': None, 'normallyInService': None, 'AdditionalEquipmentContainer': [], 

STEP 3b. Writing custom made content for the new_new_power_power_power_transformer function. Will use CP to generate the required input. After that it will be converted to dictionary and each attribute set separately. The network connection of the new component will be done as normal.

In [ ]:
# name = 'Power_Transformer_XYZ_N1'
# node_P_xfmr_1 = '646'
# node_P_xfmr_2 = '645'

# cim_profile, cim_module = get_cim_profile()
# cim:cim = cim_module

# #synchr_generator = cim.SynchronousMachine(name = name)

In [ ]:
# cim_profile, cim_module = get_cim_profile()
# cim:cim = cim_module
# xfmr_N1 = cim.PowerTransformer(name = name)

# Catalog_JSON_file_path = '../test_models/hv69_12.json' ### Power Transformer json file path
# catalog_file = '../test_models/hv69_12.json' ### Power Transformer json file path
# if catalog_file is not None:
#     print('Catalog file has content')
#     xfmr_N1_Catalogue = catalog_parser(catalog_file, network) ## used Catalog Parser and created the output Object
#     xfmr_N1.name = name
#     xfmr_N1.EquipmentContainer = container

#     for end in xfmr_N1_Catalogue.PowerTransformerEnd:
#         #end.PowerTransformer = xfmr_N1
#         number = int(end.endNumber) ## These numbers will be 1 and 2, which denotes the 1st and 2nd windings in the 2-winding transformer respectively
#         print(number)
#         terminal = cim.Terminal(name=f"{xfmr_N1.name}_t{number}", sequenceNumber=number)
#         print(terminal)
#         end.Terminal = terminal
#         xfmr_N1.Terminals.append(terminal)
#         print(xfmr_N1.Terminals)
#         network.add_to_graph(terminal)

        
#         if number == 1:
#             terminal_to_node(network, terminal, node_P_xfmr_1)
#         if number == 2:
#             terminal_to_node(network, terminal, node_P_xfmr_2)

#         List_of_Keys = list(xfmr_N1_Catalogue.PowerTransformerEnd[1].__dict__.keys())
#         List_of_Values = list(xfmr_N1_Catalogue.PowerTransformerEnd[1].__dict__.values())

#         for k in range(len(List_of_Keys)):
#             current_key = List_of_Keys[k]
#             current_value = List_of_Values[k]

#             xfmr_N1.current_key = current_value
        

# print(xfmr_N1)




Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


Catalog file has content
1
{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal", "name": "Power_Transformer_XYZ_t1", "sequenceNumber": "1"}
[{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal"}]
2
{"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal", "name": "Power_Transformer_XYZ_t2", "sequenceNumber": "2"}
[{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal"}, {"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal"}]
{"@id": "8dc87200-5717-4b10-a171-b0ae8b6a5e98", "@type": "PowerTransformer", "name": "Power_Transformer_XYZ", "EquipmentContainer": {"@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62", "@type": "Feeder"}, "Terminals": [{"@id": "17c49bd6-94e8-4d67-a228-6cf03c8fddc5", "@type": "Terminal"}, {"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal"}], "current_key": {"@id": "a54814d2-cefb-421c-8e94-4e31f85f508a", "@type": "PowerTransformerEnd"}}


In [ ]:
# ## saving to xml
# from cimgraph import utils
# utils.write_xml(network=network, filename='13_node_network_with_CP_xfmr_latest.xml')

In [ ]:
# print(xfmr_N1_Catalogue.PowerTransformerEnd[1])

{"@id": "3e00b378-b4e0-4201-9301-d96748137c77", "@type": "PowerTransformerEnd", "name": "hvmv69_12_End_2", "endNumber": "2", "grounded": "true", "rground": "0", "xground": "0", "Terminal": {"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal"}, "phaseAngleClock": "0", "connectionKind": "WindingConnection.Y", "r": "0.052092802", "ratedS": "20000000", "ratedU": "12470"}


In [ ]:
# print(list(xfmr_N1_Catalogue.PowerTransformerEnd[1].__dict__.keys())[0])

identifier


In [ ]:
# print(xfmr_N1_Catalogue.PowerTransformerEnd[1].__dict__.values())

dict_values([UUID('3e00b378-b4e0-4201-9301-d96748137c77'), '3e00b378-b4e0-4201-9301-d96748137c77', None, None, 'hvmv69_12_End_2', [], '2', 'true', '0', '0', None, None, [], None, None, None, {"@id": "87371c93-3edb-4943-8155-20cc640af7d5", "@type": "Terminal"}, [], '0', 'WindingConnection.Y', '0.052092802', '20000000', '12470', None, <cimgraph.data_profile.identity.UUID_Meta object at 0x7c08ec646200>, '{"@id": "3e00b378-b4e0-4201-9301-d96748137c77", "@type": "PowerTransformerEnd"}'])


In [ ]:
# print(len(List_of_Keys))

26


In [65]:
# for k in range(len(List_of_Keys)):
#     print(k)